# TMJDLC — GoFile Edition

Private Telegram controller + Javtiful channel/video downloader + public GoFile folders (20 videos per folder). Supports channel page selection and up to 2 simultaneous downloads + 2 simultaneous GoFile uploads. Telegram receives links only. No database, persistent queue or duplicate detection.


## 1. Configure Colab Secrets

- `TG_BOT_TOKEN` — Telegram bot token
- `TG_OWNER_ID` — your numeric Telegram user ID
- `GOFILE_API_TOKEN` — optional; leave unset to create a guest account

Enable notebook access to these secrets. The guest token is saved in `data/gofile_state.json`; keep that file if you want to reuse the same guest account after restarting the notebook.


In [ ]:
#@title 2. Clone or update the gofile branch
%cd /content
import os, subprocess
repo = '/content/TMJDLC'
if os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git', '-C', repo, 'fetch', 'origin', 'gofile'], check=True)
    subprocess.run(['git', '-C', repo, 'checkout', 'gofile'], check=True)
    subprocess.run(['git', '-C', repo, 'reset', '--hard', 'origin/gofile'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'gofile', 'https://github.com/Telebotfaroff/TMJDLC.git', repo], check=True)
%cd /content/TMJDLC
print('Checked out branch gofile:', subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip())


In [ ]:
#@title 3. Install lightweight dependencies
%cd /content/TMJDLC
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-gofile.txt'], check=True)
subprocess.run(['bash','-lc','apt-get update -qq && apt-get install -y ffmpeg'], check=True)
subprocess.run([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'], check=True)
subprocess.run([sys.executable, '-m', 'compileall', '-q', 'gofile_bot', 'tmjdlc/providers/javtiful', 'tmjdlc/downloaders'], check=True)
print('Dependencies installed; Python compilation passed.')


In [ ]:
#@title 4. Load secrets and prepare local state
import os
from pathlib import Path
from google.colab import userdata
required = ['TG_BOT_TOKEN', 'TG_OWNER_ID']
missing = []
for key in required:
    value = userdata.get(key)
    if not value: missing.append(key)
    else: os.environ[key] = str(value).strip()
if missing: raise RuntimeError('Missing Colab Secrets: ' + ', '.join(missing))
try: gofile_token = userdata.get('GOFILE_API_TOKEN')
except Exception: gofile_token = None
if gofile_token: os.environ['GOFILE_API_TOKEN'] = str(gofile_token).strip()
os.environ['TMJDLC_WORK_DIR'] = '/content/tmjdlc-gofile'
os.environ['TMJDLC_STATE_FILE'] = '/content/TMJDLC/data/gofile_state.json'
Path(os.environ['TMJDLC_WORK_DIR']).mkdir(parents=True, exist_ok=True)
Path('/content/TMJDLC/data').mkdir(parents=True, exist_ok=True)
print('Telegram credentials loaded. GoFile token supplied:', bool(gofile_token))
print('State file:', os.environ['TMJDLC_STATE_FILE'])


In [ ]:
#@title 5. Start TMJDLC GoFile bot (keep this cell running)
%cd /content/TMJDLC
import subprocess, sys, os
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
subprocess.run([sys.executable, '-u', '-m', 'gofile_bot.app'], env=env, check=True)
